# 03 — Spatial Hazard Index
**HHE Lab Sardinia · Marine Litter Hazard Assessment**

Inputs → `data/processed/beach_litter.csv` · `floating_litter.csv`  
Outputs → `data/processed/hazard_grid.csv` · `data/sardinia_hazard_index.html`

**Method:**
1. Min-max normalize monitoring scores per compartment (0–1)
2. IDW interpolation onto 10km grid covering Sardinia + 20km buffer
3. Composite hazard index = mean(beach_score, floating_score) per cell
4. 5-class classification: Very Low → Very High

**Limitations:** 6 beach stations + 14 floating stations → sparse coverage,
IDW smooths toward station values. Seafloor compartment missing (no MSFD module available).

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import folium
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
from shapely.geometry import Point, box
from scipy.spatial import cKDTree
from geodatasets import get_path
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')

ROOT = Path('..').resolve()
OUT  = ROOT / 'data' / 'processed'
MAPS = ROOT / 'data'
FIGS = ROOT / 'data' / 'figures'
FIGS.mkdir(exist_ok=True)

beach    = pd.read_csv(OUT / 'beach_litter.csv')
floating = pd.read_csv(OUT / 'floating_litter.csv')
print('Data loaded.')

## 1. Sardinia polygon (Natural Earth 110m)

In [ ]:
sardinia_box  = box(8.1, 38.86, 9.87, 41.25)
land          = gpd.read_file(get_path('naturalearth.land'))
sardinia_gdf  = land.clip(sardinia_box)
sardinia_poly = sardinia_gdf.geometry.iloc[0]
if sardinia_poly.geom_type == 'MultiPolygon':
    sardinia_poly = max(sardinia_poly.geoms, key=lambda p: p.area)
print(f'Centroid: {sardinia_poly.centroid.x:.2f}, {sardinia_poly.centroid.y:.2f}')

## 2. Station aggregation & normalization

In [ ]:
beach_st = (
    beach.groupby('beach_id').agg(
        lat=('lat','first'), lon=('lon','first'),
        name=('beach_name','first'),
        items_100m=('items_per_100m','mean'),
        n_surveys=('survey_id','count')
    ).reset_index().dropna(subset=['lat','lon'])
)
float_st = (
    floating.groupby('station_id').agg(
        lat=('lat_station','first'), lon=('lon_station','first'),
        n_obs=('station_id','count'),
        plastic_pct=('material', lambda x: (x=='Artificial polymer').sum()/len(x)*100)
    ).reset_index().dropna(subset=['lat','lon'])
)

def minmax(s):
    return (s - s.min()) / (s.max() - s.min() + 1e-10)

beach_st['score'] = minmax(beach_st['items_100m'])
float_st['score'] = minmax(float_st['n_obs'])

print('Beach stations:')
print(beach_st[['name','items_100m','score']].sort_values('score', ascending=False).round(3).to_string(index=False))

## 3. Coastal grid + IDW interpolation

In [ ]:
# 0.12° ≈ 10km grid
lons = np.arange(8.1, 9.9, 0.12)
lats = np.arange(38.86, 41.25, 0.12)
lon_g, lat_g = np.meshgrid(lons, lats)
lon_f, lat_f = lon_g.ravel(), lat_g.ravel()

sardinia_buf = sardinia_poly.buffer(0.25)
mask = np.array([sardinia_buf.contains(Point(lo, la)) for lo, la in zip(lon_f, lat_f)])
lon_c, lat_c = lon_f[mask], lat_f[mask]
print(f'Grid cells: {mask.sum()}')

def idw(g_lons, g_lats, s_lons, s_lats, scores, power=2, radius=1.0):
    result = np.full(len(g_lons), np.nan)
    tree = cKDTree(np.column_stack([s_lons, s_lats]))
    for i, (glo, gla) in enumerate(zip(g_lons, g_lats)):
        idxs = tree.query_ball_point([glo, gla], radius)
        if not idxs: continue
        d = np.sqrt((s_lons[idxs]-glo)**2 + (s_lats[idxs]-gla)**2)
        w = 1 / (d**power + 1e-9)
        result[i] = np.sum(w * scores[idxs]) / np.sum(w)
    return result

beach_interp = idw(lon_c, lat_c, beach_st['lon'].values, beach_st['lat'].values,
                   beach_st['score'].values, radius=1.0)
float_interp = idw(lon_c, lat_c, float_st['lon'].values, float_st['lat'].values,
                   float_st['score'].values, radius=1.2)

hazard = np.nanmean(np.stack([beach_interp, float_interp], axis=1), axis=1)

grid_df = pd.DataFrame({
    'lon': lon_c, 'lat': lat_c,
    'beach_score': beach_interp, 'float_score': float_interp,
    'hazard_index': hazard
}).dropna(subset=['hazard_index'])

grid_df['hazard_class'] = pd.cut(
    grid_df['hazard_index'], bins=[0,.2,.4,.6,.8,1.],
    labels=['Very Low','Low','Medium','High','Very High'], include_lowest=True
)
grid_df.to_csv(OUT / 'hazard_grid.csv', index=False)
grid_df['hazard_class'].value_counts().sort_index()

## 4. Static 3-panel map

In [ ]:
cmap = LinearSegmentedColormap.from_list('hazard',
       ['#27ae60','#f1c40f','#e67e22','#e74c3c','#8e44ad'])

fig, axes = plt.subplots(1, 3, figsize=(18, 9))
for ax, title, col, st_df, st_marker in zip(
    axes,
    ['Beach Litter Score','Floating Litter Score','Composite Hazard Index'],
    ['beach_score','float_score','hazard_index'],
    [beach_st, float_st, None],
    ['o','^',None]
):
    sc = ax.scatter(grid_df['lon'], grid_df['lat'],
                    c=grid_df[col], cmap=cmap, s=55, alpha=0.85,
                    vmin=0, vmax=1, edgecolors='none', marker='s')
    cx, cy = sardinia_poly.exterior.xy
    ax.plot(cx, cy, 'k-', linewidth=0.8, alpha=0.5)
    if col in ('beach_score','hazard_index'):
        ax.scatter(beach_st['lon'], beach_st['lat'], marker='o',
                   s=60, c='white', edgecolors='black', lw=1.5, zorder=5)
    if col in ('float_score','hazard_index'):
        ax.scatter(float_st['lon'], float_st['lat'], marker='^',
                   s=60, c='white', edgecolors='black', lw=1.5, zorder=5)
    plt.colorbar(sc, ax=ax, fraction=0.03, pad=0.04)
    ax.set_title(title, fontweight='bold', pad=8)
    ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
    ax.set_xlim(8.0, 10.0); ax.set_ylim(38.7, 41.4)

legend_el = [
    mpatches.Patch(color='#27ae60', label='Very Low (0–0.2)'),
    mpatches.Patch(color='#f1c40f', label='Low (0.2–0.4)'),
    mpatches.Patch(color='#e67e22', label='Medium (0.4–0.6)'),
    mpatches.Patch(color='#e74c3c', label='High (0.6–0.8)'),
    mpatches.Patch(color='#8e44ad', label='Very High (0.8–1.0)'),
]
axes[2].legend(handles=legend_el, loc='lower right', fontsize=8, framealpha=0.9)
fig.suptitle('Marine Litter Hazard Index — Sardinia (2020–2023)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(FIGS / 'hazard_index_map.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Interactive hazard map → `sardinia_hazard_index.html`

In [ ]:
def hcolor(v):
    if v < 0.2: return '#27ae60'
    if v < 0.4: return '#f1c40f'
    if v < 0.6: return '#e67e22'
    if v < 0.8: return '#e74c3c'
    return '#8e44ad'

m = folium.Map(location=[40.1, 9.0], zoom_start=7, tiles='CartoDB positron')
cell = 0.06

for layer_name, col, show in [
    ('🗺️ Hazard Index (composite)', 'hazard_index', True),
    ('🏖️ Beach Score', 'beach_score', False),
    ('🌊 Floating Score', 'float_score', False)
]:
    grp = folium.FeatureGroup(name=layer_name, show=show)
    for _, row in grid_df.dropna(subset=[col]).iterrows():
        v = row[col]
        folium.Rectangle(
            bounds=[[row['lat']-cell, row['lon']-cell],[row['lat']+cell, row['lon']+cell]],
            color=None, fill=True, fill_color=hcolor(v), fill_opacity=0.6,
            tooltip=(f"Hazard: {row['hazard_index']:.2f} ({row['hazard_class']}) | "
                     f"Beach: {row['beach_score']:.2f} | Float: {row['float_score']:.2f}")
        ).add_to(grp)
    grp.add_to(m)

# Station markers
st_grp = folium.FeatureGroup(name='📍 Stations', show=True)
for _, r in beach_st.iterrows():
    folium.CircleMarker(
        location=[r['lat'], r['lon']], radius=10,
        color='white', weight=2.5,
        fill=True, fill_color=hcolor(r['score']), fill_opacity=0.95,
        popup=folium.Popup(
            f"<b>{r['name']}</b><br>items/100m: <b>{r['items_100m']:.1f}</b><br>"
            f"score: <b>{r['score']:.2f}</b>", max_width=200),
        tooltip=f"🏖️ {r['name']}: {r['items_100m']:.0f} items/100m"
    ).add_to(st_grp)
for _, r in float_st.iterrows():
    folium.RegularPolygonMarker(
        location=[r['lat'], r['lon']], number_of_sides=3, radius=9,
        color='white', weight=2,
        fill=True, fill_color=hcolor(r['score']), fill_opacity=0.9,
        popup=folium.Popup(
            f"<b>{r['station_id']}</b><br>obs: <b>{r['n_obs']}</b><br>"
            f"plastic: <b>{r['plastic_pct']:.1f}%</b><br>score: <b>{r['score']:.2f}</b>",
            max_width=200),
        tooltip=f"🌊 {r['station_id']}: {r['n_obs']} obs"
    ).add_to(st_grp)
st_grp.add_to(m)

legend_html = """
<div style='position:fixed;bottom:40px;left:40px;z-index:1000;background:white;
     padding:14px 18px;border-radius:8px;box-shadow:2px 2px 8px rgba(0,0,0,0.2);
     font-family:Arial,sans-serif;font-size:12px'>
  <b style='font-size:14px'>Hazard Index</b><br>
  <small style='color:#888'>MSFD D10 · Sardinia · 2020–2023</small>
  <hr style='margin:8px 0;border-color:#eee'>
  <span style='background:#27ae60;padding:2px 8px;border-radius:3px;color:white'>Very Low</span> &lt;0.2<br><br>
  <span style='background:#f1c40f;padding:2px 8px;border-radius:3px'>Low</span> 0.2–0.4<br><br>
  <span style='background:#e67e22;padding:2px 8px;border-radius:3px;color:white'>Medium</span> 0.4–0.6<br><br>
  <span style='background:#e74c3c;padding:2px 8px;border-radius:3px;color:white'>High</span> 0.6–0.8<br><br>
  <span style='background:#8e44ad;padding:2px 8px;border-radius:3px;color:white'>V.High</span> &gt;0.8
  <hr style='margin:8px 0;border-color:#eee'>
  ● beach &nbsp; ▲ floating<br>
  <small style='color:#777'>IDW · 10km grid · 2 compartments</small>
</div>"""
m.get_root().html.add_child(folium.Element(legend_html))
folium.LayerControl(collapsed=False, position='topright').add_to(m)
m.save(str(MAPS / 'sardinia_hazard_index.html'))
print('Saved: sardinia_hazard_index.html')

## 6. Hotspot analysis

In [ ]:
print('Hazard class distribution:')
print(grid_df['hazard_class'].value_counts().sort_index())
print()
print('Top 10 hotspot cells:')
print(grid_df.nlargest(10,'hazard_index')[['lon','lat','beach_score','float_score','hazard_index','hazard_class']].round(3).to_string(index=False))
print()
print('Beach station ranking:')
print(beach_st[['name','items_100m','score']].sort_values('score',ascending=False).round(3).to_string(index=False))